In [1]:
!pip install gradio -q

import gradio as gr
import numpy as np
import librosa
import pickle
import os
import random
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# --- Konstanter ---
SR         = 22050
N_MELS     = 128
N_FFT      = 2048
HOP_LENGTH = 512
TARGET_LEN = SR * 3

# --- Pakk ut datasett hvis nødvendig ---
import zipfile
ZIP_PATH  = Path("/content/drive/MyDrive/IRMAS-TrainingData.zip")
DATA_DIR  = Path("/content/IRMAS-TrainingData")

if not DATA_DIR.exists():
    print("Pakker ut datasett...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content/")
    print("Ferdig!")

# --- Last inn modell og label encoder ---
PROCESSED = Path("/content/drive/MyDrive/processed")
model     = keras.models.load_model(str(PROCESSED / "improved_best.keras"))

with open(PROCESSED / "label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print(f"Modell lastet! Klasser: {list(le.classes_)}")

# --- Hent 3 tilfeldige eksempelfiler per klasse ---
CLASSES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
examples = []
for cls in CLASSES:
    wav_files = list((DATA_DIR / cls).glob("*.wav"))
    samples   = random.sample(wav_files, min(3, len(wav_files)))
    for f in samples:
        examples.append([str(f)])

print(f"Lastet {len(examples)} eksempelfiler")

# --- Preprocessing ---
def preprocess_audio(audio_path):
    y, _ = librosa.load(audio_path, sr=SR, mono=True)
    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)))
    else:
        y = y[:TARGET_LEN]
    S      = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS,
                                            n_fft=N_FFT, hop_length=HOP_LENGTH)
    S_db   = librosa.power_to_db(S, ref=np.max)
    S_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-6)
    return S_norm[np.newaxis, ..., np.newaxis]

# --- Prediksjonsfunksjon ---
INSTRUMENT_NAMES = {
    "cel": "Cello",
    "cla": "Klarinett",
    "flu": "Fløyte",
    "gac": "Akustisk gitar",
    "gel": "Elektrisk gitar",
    "org": "Orgel",
    "pia": "Piano",
    "sax": "Saksofon",
    "tru": "Trompet",
    "vio": "Fiolin",
    "voi": "Sang"
}

def classify_instrument(audio_path):
    if audio_path is None:
        return {}
    try:
        spec  = preprocess_audio(audio_path)
        probs = model.predict(spec, verbose=0)[0]
        result = {INSTRUMENT_NAMES.get(le.classes_[i], le.classes_[i]): float(probs[i])
                  for i in range(len(le.classes_))}
        return result
    except Exception as e:
        return {"Feil": str(e)}

# --- Gradio-grensesnitt ---
demo = gr.Interface(
    fn=classify_instrument,
    inputs=gr.Audio(
        type="filepath",
        label="Last opp lydfil eller velg et eksempel nedenfor"
    ),
    outputs=gr.Label(
        num_top_classes=5,
        label="Instrument (topp 5)"
    ),
    title="Instrument Classifier",
    description="""
    Klassifiserer musikkinstrumenter fra korte lydklipp ved hjelp av en CNN trent på IRMAS-datasettet.
    Last opp en egen lydfil, eller velg ett av eksemplene nedenfor.
    Støtter 11 klasser: cello, klarinett, fløyte, akustisk gitar, elektrisk gitar, orgel, piano, saksofon, trompet, fiolin og sang.
    """,
    examples=examples,
    cache_examples=False,
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Mounted at /content/drive
Pakker ut datasett...
Ferdig!


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Modell lastet! Klasser: [np.str_('cel'), np.str_('cla'), np.str_('flu'), np.str_('gac'), np.str_('gel'), np.str_('org'), np.str_('pia'), np.str_('sax'), np.str_('tru'), np.str_('vio'), np.str_('voi')]
Lastet 33 eksempelfiler
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3bafc567f270bf92b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
